**Group Name:** Course Project Group 25

**Members:** 汪鸿源 72510222

# AIGC Detection - Test Inference

 **Note**: This notebook is used to run inference on the test set and generate prediction results.The model.pth file is placed in the same directory.

In [ ]:
import os
# Ban version check of albumentations
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
# hf-mirror.com configuration 
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import cv2
import glob
import json
from tqdm import tqdm
from transformers import ViTModel, ViTConfig
from torchvision.models import resnet50, ResNet50_Weights
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch.nn.functional as F

# CONFIGURATIONS
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TEST_DIR = r'/root/autodl-tmp/5489project/Test_set'
CHECKPOINT_DIR = r'/root/autodl-tmp/5489project/checkpoints'

print(f"Using device: {DEVICE}")
print(f"Test directory: {TEST_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

# --- Model Definitions (Must match Train.ipynb) ---
class ViTForAIGC(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()
        model_name = "google/vit-base-patch16-224-in21k"
        self.vit = ViTModel.from_pretrained(model_name) if pretrained else ViTModel(ViTConfig.from_pretrained(model_name))

        if pretrained:
            for param in self.vit.parameters():
                param.requires_grad = False
            for i, layer in enumerate(self.vit.encoder.layer):
                if i >= 8: 
                    for param in layer.parameters():
                        param.requires_grad = True
        
        for param in self.vit.layernorm.parameters():
            param.requires_grad = True

        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.Linear(self.vit.config.hidden_size, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, rgb_images: torch.Tensor):
        outputs = self.vit(pixel_values=rgb_images)
        cls_token_output = outputs.last_hidden_state[:, 0, :]
        cls_token_output = self.dropout(cls_token_output)
        logits = self.classifier(cls_token_output)
        return {"logits": logits}

class ResNetForAIGC(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()
        # Use standard ResNet-50
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        self.resnet = resnet50(weights=weights)

        # Freeze layers if pretrained
        if pretrained:
            for param in self.resnet.parameters():
                param.requires_grad = False
            # Unfreeze the last block (layer4) and fully connected layer
            for param in self.resnet.layer4.parameters():
                param.requires_grad = True
        
        # Replace the final fully connected layer
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, num_classes)
        # Initialize the new fc layer
        nn.init.xavier_uniform_(self.resnet.fc.weight)
        nn.init.zeros_(self.resnet.fc.bias)

    def forward(self, rgb_images: torch.Tensor):
        logits = self.resnet(rgb_images)
        return {"logits": logits}

# Define transforms 
test_transform_rgb = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

def preprocess_image(image_path: str):
    # 1. Load Image using cv2 
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Failed to load image: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 2. Transform
    augmented = test_transform_rgb(image=image)
    rgb_tensor = augmented['image'].unsqueeze(0).to(DEVICE)
    
    return rgb_tensor

def run_inference(model_name, checkpoint_path, output_csv):
    print(f"\n{'='*20} Processing {model_name} {'='*20}")
    
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        return False

    # Initialize Model
    print(f"Initializing {model_name}...")
    model = None
    try:
        if model_name == 'ViT-Base':
            model = ViTForAIGC(num_classes=2, pretrained=False)
        elif model_name == 'ResNet50':
            model = ResNetForAIGC(num_classes=2, pretrained=False)
        else:
            raise ValueError(f"Unknown model name: {model_name}")
    except Exception as e:
        print(f"Warning: Could not initialize with pretrained=False. Trying pretrained=True. Error: {e}")
        if model_name == 'ViT-Base':
            model = ViTForAIGC(num_classes=2, pretrained=True)
        elif model_name == 'ResNet50':
            model = ResNetForAIGC(num_classes=2, pretrained=True)

    if model is None:
        print("Failed to initialize model.")
        return False

    # Load weights
    print(f"Loading weights from {checkpoint_path}...")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        else:
            state_dict = checkpoint
        model.load_state_dict(state_dict)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return False

    model = model.to(DEVICE)
    model.eval()
    
    # Retrieve images
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff')
    test_images = []
    for ext in exts:
        test_images.extend(glob.glob(os.path.join(TEST_DIR, ext)))
    
    if not test_images:
        print(f"No images found in {TEST_DIR}.")
        return False
    else:
        print(f"Found {len(test_images)} images.")
        
    predictions = []
    filenames = []
    
    with torch.no_grad():
        for img_path in tqdm(test_images, desc=f'Inference {model_name}'):
            try:
                rgb_tensor = preprocess_image(img_path)
                outputs = model(rgb_tensor)
                logits = outputs['logits']
                probs = F.softmax(logits, dim=1)[0]
                fake_prob = probs[1].item()
                
                filename = os.path.basename(img_path)
                predictions.append(fake_prob) 
                filenames.append(filename)
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                filename = os.path.basename(img_path)
                predictions.append(0.5)
                filenames.append(filename)

    results_df = pd.DataFrame({
        'ID': filenames,
        'prob': predictions
    })
    results_df['ID'] = results_df['ID'].apply(lambda x: os.path.splitext(x)[0])
    results_df = results_df.sort_values(by='ID').reset_index(drop=True)
    results_df.to_csv(output_csv, index=False)
    print(f"Saved predictions to {output_csv}")
    return True

def ensemble_predictions(resnet_csv, vit_csv, output_csv, w_resnet=0.4, w_vit=0.6): # Adjusted weighted average
    print(f"\n{'='*20} Running Ensemble {'='*20}")
    if not os.path.exists(resnet_csv) or not os.path.exists(vit_csv):
        print("One or both probability files not found. Skipping ensemble.")
        return

    print(f"Ensembling {resnet_csv} and {vit_csv}...")
    df_resnet = pd.read_csv(resnet_csv)
    df_vit = pd.read_csv(vit_csv)

    df_resnet = df_resnet.sort_values('ID').reset_index(drop=True)
    df_vit = df_vit.sort_values('ID').reset_index(drop=True)

    if not df_resnet['ID'].equals(df_vit['ID']):
        print("Error: IDs do not match between files.")
        return

    df_final = df_resnet.copy()
    # Weighted Average
    df_final['prob'] = w_resnet * df_resnet['prob'] + w_vit * df_vit['prob']
    # Convert to label
    df_final['label'] = (df_final['prob'] > 0.5).astype(int)
    
    submission_df = df_final[['ID', 'label']]
    submission_df.to_csv(output_csv, index=False)
    print(f"Ensemble submission saved to {output_csv}")
    print(f"Prediction distribution: {dict(submission_df['label'].value_counts())}")


Using device: cuda
Test directory: /root/autodl-tmp/5489project/Test_set
Checkpoint directory: /root/autodl-tmp/5489project/checkpoints


In [2]:
# 1. Run ResNet50
resnet_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model_ResNet50.pth')
resnet_out = 'submission_test_ResNet50_prob.csv'
run_inference('ResNet50', resnet_ckpt, resnet_out)
    
# 2. Run ViT-Base
vit_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model_ViT-Base.pth')
vit_out = 'submission_test_ViT-Base_prob.csv'
run_inference('ViT-Base', vit_ckpt, vit_out)
    
# 3. Ensemble
ensemble_out = 'submission_ensemble.csv'
ensemble_predictions(resnet_out, vit_out, ensemble_out)


==================== Processing ResNet50 ====================
Initializing ResNet50...
Loading weights from /root/autodl-tmp/5489project/checkpoints/best_model_ResNet50.pth...
Loading weights from /root/autodl-tmp/5489project/checkpoints/best_model_ResNet50.pth...
Weights loaded successfully.
Found 2500 images.
Weights loaded successfully.
Found 2500 images.


Inference ResNet50: 100%|██████████| 2500/2500 [00:09<00:00, 271.27it/s]



Saved predictions to submission_test_ResNet50_prob.csv

==================== Processing ViT-Base ====================
Initializing ViT-Base...
Loading weights from /root/autodl-tmp/5489project/checkpoints/best_model_ViT-Base.pth...
Loading weights from /root/autodl-tmp/5489project/checkpoints/best_model_ViT-Base.pth...
Weights loaded successfully.
Found 2500 images.
Weights loaded successfully.
Found 2500 images.


Inference ViT-Base: 100%|██████████| 2500/2500 [00:12<00:00, 204.03it/s]

Saved predictions to submission_test_ViT-Base_prob.csv

==================== Running Ensemble ====================
Ensembling submission_test_ResNet50_prob.csv and submission_test_ViT-Base_prob.csv...
Ensemble submission saved to submission_ensemble.csv
Prediction distribution: {1: np.int64(1477), 0: np.int64(1023)}
